In [40]:
pip install seaborn

Note: you may need to restart the kernel to use updated packages.


In [41]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# 1- Setup random numbers when initializing values 



In [42]:
# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# 2- Device used for training

In [43]:
# Set device (GPU if available, else CPU)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


# 3- Class for messages to handle tokenization, length

In [44]:
class MessageDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=True,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'token_type_ids': encoding['token_type_ids'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


# 4- Load csv with the cleaned messages

In [45]:
def load_and_prepare_data(csv_path):
    """Load and prepare data from a CSV file."""
    print(f"Loading data from {csv_path}...")
    df = pd.read_csv(csv_path)
    df=df.head(1800)
    # Print first few rows to verify data
    print("First few rows of the data:")
    print(df.head())
    
    # Check distribution of topics
    print("\nDistribution of topics:")
    print(df['topic'].value_counts())
    
    # Map topic names to numerical values
    topic_mapping = {name: idx for idx, name in enumerate(df['topic'].unique())}
    reverse_mapping = {idx: name for name, idx in topic_mapping.items()}
    
    print("\nTopic to index mapping:")
    for topic, idx in topic_mapping.items():
        print(f"{topic} -> {idx}")
    
    # Create labels from topics
    df['label'] = df['topic'].map(topic_mapping)
    
    # Split data into train and validation sets
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=RANDOM_SEED, stratify=df['label'])
    
    print(f"\nTraining set size: {len(train_df)}")
    print(f"Validation set size: {len(val_df)}")
    
    return train_df, val_df, topic_mapping, reverse_mapping

# 5- Seperate data, into training and validation.
#    Shuffles the data as well.

In [46]:
def create_data_loaders(train_df, val_df, tokenizer, batch_size=16):
    """Create PyTorch DataLoaders for training and validation."""
    train_dataset = MessageDataset(
        texts=train_df['messages'].values,
        labels=train_df['label'].values,
        tokenizer=tokenizer
    )
    
    val_dataset = MessageDataset(
        texts=val_df['messages'].values,
        labels=val_df['label'].values,
        tokenizer=tokenizer
    )
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )
    
    return train_loader, val_loader

# Define epoch and update model as needed

In [47]:
def train_epoch(model, data_loader, optimizer, scheduler, device):
    """Train the model for one epoch."""
    model.train() #Put model in training mode
    losses = []
    correct_predictions = 0
    total_predictions = 0
    
    progress_bar = tqdm(data_loader, desc="Training") ##creates progress bar, just for visuals
    
    ##Load training data
    for batch in progress_bar: 
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        #Load data from messages class and make prediction
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels
        )
        
        #calculate los and get raw prediction
        loss = outputs.loss
        logits = outputs.logits
        
        _, preds = torch.max(logits, dim=1) #transform prediction to value
        
        #check prediction
        correct_predictions += torch.sum(preds == labels)
        total_predictions += len(labels)
        
        losses.append(loss.item())
        
        #Backpropagation to make changes to nodes for better predictions
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': f"{loss.item():.4f}",
            'acc': f"{(correct_predictions/total_predictions):.4f}"
        })
    
    return correct_predictions.double() / total_predictions, np.mean(losses)

# Evaluate Epoch with Validation values, and set accuracy

In [49]:
def eval_model(model, data_loader, device):
    """Evaluate the model on validation data."""
    model.eval()
    losses = []
    correct_predictions = 0
    total_predictions = 0
    
    all_labels = []
    all_predictions = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                labels=labels
            )
            
            loss = outputs.loss
            logits = outputs.logits
            
            _, preds = torch.max(logits, dim=1)
            
            correct_predictions += torch.sum(preds == labels)
            total_predictions += len(labels)
            
            losses.append(loss.item())
            
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(preds.cpu().numpy())
    
    return (
        correct_predictions.double() / total_predictions,
        np.mean(losses),
        all_labels,
        all_predictions
    )


In [50]:
def plot_confusion_matrix(y_true, y_pred, reverse_mapping):
    """Plot confusion matrix."""
    plt.figure(figsize=(10, 8))
    cm = confusion_matrix(y_true, y_pred)
    
    # Convert indices to topic names using reverse_mapping
    labels = [reverse_mapping[i] for i in range(len(reverse_mapping))]
    
    sns.heatmap(
        cm, 
        annot=True, 
        fmt='d', 
        cmap='Blues',
        xticklabels=labels,
        yticklabels=labels
    )
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png')
    plt.close()

# Train and Evaluate and save best model


In [51]:

def train_and_evaluate(model, train_loader, val_loader, optimizer, scheduler, device, epochs=4):
    """Train and evaluate the model."""
    best_accuracy = 0
    training_history = {'train_acc': [], 'train_loss': [], 'val_acc': [], 'val_loss': []}
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")
        
        # Train the model
        train_acc, train_loss = train_epoch(
            model=model,
            data_loader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device
        )
        
        print(f"Train loss: {train_loss:.4f}, accuracy: {train_acc:.4f}")
        
        # Evaluate the model
        val_acc, val_loss, val_labels, val_preds = eval_model(
            model=model,
            data_loader=val_loader,
            device=device
        )
        
        print(f"Validation loss: {val_loss:.4f}, accuracy: {val_acc:.4f}")
        
        # Save training history
        training_history['train_acc'].append(train_acc.item())
        training_history['train_loss'].append(train_loss)
        training_history['val_acc'].append(val_acc.item())
        training_history['val_loss'].append(val_loss)
        
        # Save best model
        if val_acc > best_accuracy:
            torch.save(model.state_dict(), 'best_model.pt')
            best_accuracy = val_acc
            print(f"Best model saved with accuracy: {best_accuracy:.4f}")
    
    return training_history, val_labels, val_preds

# Plot training history

In [52]:
def plot_training_history(history):
    """Plot training history."""
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(history['train_acc'], label='train')
    plt.plot(history['val_acc'], label='validation')
    plt.title('Accuracy')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history['train_loss'], label='train')
    plt.plot(history['val_loss'], label='validation')
    plt.title('Loss')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.close()


# For testing purpose, evaluate model and return predicted topic

In [53]:
def predict_topic(text, model, tokenizer, topic_mapping):
    """Predict the topic of a given text."""
    # Create a reverse mapping from index to topic
    reverse_mapping = {idx: topic for topic, idx in topic_mapping.items()}
    
    # Prepare the text using tokenizer
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=128,
        return_token_type_ids=True,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
    )
    
    # Move tensors to the right device
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    token_type_ids = encoding['token_type_ids'].to(device)
    
    # Set model to evaluation mode
    model.eval()
    
    # Get prediction
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        _, preds = torch.max(outputs.logits, dim=1)
    
    # Convert prediction to topic name
    predicted_topic = reverse_mapping[preds.item()]
    
    return predicted_topic


# Put all the functions together

In [54]:
def main():
    """Main function to train and evaluate the BERT model for topic classification."""
    # 1. Load and prepare data
    csv_path = './GMO_Cleaned_Messages_Final.csv'  # Replace with your CSV file path
    train_df, val_df, topic_mapping, reverse_mapping = load_and_prepare_data(csv_path)
    
    # 2. Initialize BERT tokenizer
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    
    # 3. Create data loaders
    train_loader, val_loader = create_data_loaders(train_df, val_df, tokenizer)
    
    # 4. Initialize BERT model for sequence classification (with 5 topics)
    model = BertForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        num_labels=len(topic_mapping),
        output_attentions=False,
        output_hidden_states=False
    )
    model = model.to(device)
    
    # 5. Initialize optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=2e-5)
    total_steps = len(train_loader) * 4  # 4 epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )
    
    # 6. Train and evaluate the model
    history, val_labels, val_preds = train_and_evaluate(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device
    )
    
    # 7. Plot training history
    plot_training_history(history)
    
    # 8. Plot confusion matrix
    plot_confusion_matrix(val_labels, val_preds, reverse_mapping)
    
    # 9. Print classification report
    print("\nClassification Report:")
    target_names = [reverse_mapping[i] for i in range(len(reverse_mapping))]
    print(classification_report(val_labels, val_preds, target_names=target_names))
    
    # 10. Load best model for prediction
    best_model = BertForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        num_labels=len(topic_mapping)
    )
    best_model.load_state_dict(torch.load('best_model.pt'))
    best_model = best_model.to(device)
    
    # 11. Example of prediction
    example_texts = [
        "I just can't seem to stay away from iquor stores when i come back from work. It helps with the pain",
        "I feel like my house is so disorganized, my family is fighting constantly and I cant think of anything other than leaving",
        "I want to learn more about Jesus",
        "i just dont want to live anymore. i want the pain to go away",
        "Has anyone tried the new update for the operating system? Any issues?"
    ]
    
    print("\nExample Predictions:")
    for text in example_texts:
        predicted_topic = predict_topic(text, best_model, tokenizer, topic_mapping)
        print(f"Text: {text[:50]}... | Predicted Topic: {predicted_topic}")

if __name__ == "__main__":
    main()

Loading data from ./GMO_Cleaned_Messages_Final.csv...
First few rows of the data:
                           contact_uuid first_name last_name        source  \
0  57d33318-8dad-4f74-bba9-5baf5b1d6c82      Sarah     Jones       Website   
1  fac69206-6fcb-4f4f-81dc-b2750948f2f0     Nathan  Anderson       Website   
2  8e008cea-799a-4aa5-ab05-f81e5f75f806       Ryan    Martin  Social Media   
3  f2f99a14-b354-4c6f-80eb-ddcd6971f97d      David     Brown  Social Media   
4  d90ff358-b4f6-4822-bf73-fd321822d4ee     Ashley     Davis  Social Media   

  language_code                        decision        country     region  \
0            en   I do not want to follow Jesus  United States    Ontario   
1            en  I just decided to follow Jesus   South Africa  Cape Town   
2            en    I want to come back to Jesus         Canada     Sydney   
3            en  I just decided to follow Jesus      Australia  Cape Town   
4            en       I am not sure about Jesus   South Africa  

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Epoch 1/4


Training: 100%|██████████| 90/90 [08:30<00:00,  5.67s/it, loss=1.2674, acc=0.4021]


Train loss: 1.5146, accuracy: 0.4021


...

Train loss: 0.2460, accuracy: 0.9444


Evaluating: 100%|██████████| 23/23 [00:30<00:00,  1.34s/it]


Validation loss: 0.4443, accuracy: 0.8528
Best model saved with accuracy: 0.8528

Classification Report:
                               precision    recall  f1-score   support

Alcoholism and Drug Addiction       0.86      0.80      0.83        60
                       Crisis       0.70      0.76      0.73        59
          Family and Marriage       0.85      0.78      0.82        51
                      General       0.92      0.86      0.89        77
            Spiritual Warfare       0.95      0.97      0.96        77
                      Suicide       0.79      0.92      0.85        36

                     accuracy                           0.85       360
                    macro avg       0.84      0.85      0.84       360
                 weighted avg       0.86      0.85      0.85       360



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Example Predictions:
Text: I just can't seem to stay away from iquor stores w... | Predicted Topic: Alcoholism and Drug Addiction
Text: I feel like my house is so disorganized, my family... | Predicted Topic: Family and Marriage
Text: I want to learn more about Jesus... | Predicted Topic: General
Text: i just dont want to live anymore. i want the pain ... | Predicted Topic: Suicide
Text: Has anyone tried the new update for the operating ... | Predicted Topic: General
